# Skin: is the training-data effect symmetric across sources? (exploratory)

Follows `data/derm/derm_source_symmetry_plan.json`. Splits different-architecture pairs into HAM10000-only, BCN20000-only and mixed pairs, and compares their shared-miss ratios (HIBA melanoma, HIBA any cancer, PAD any cancer). Same referral rule and patient bootstrap as the locked skin analysis. Output: `data/derm/derm_source_symmetry_results.json`. CPU; about 10 minutes.

In [1]:
# Cell 1: load
from google.colab import drive
drive.mount('/content/drive')
import os, json, datetime, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
DERM = '/content/drive/MyDrive/stochastic_orthogonality/data/derm'
CLASSES8 = ['MEL', 'BCC', 'SCC', 'AK', 'NV', 'BKL', 'DF', 'VASC']
ARCHS = ['resnet50', 'densenet121', 'efficientnet_b3', 'convnext_t', 'vit_b', 'vit_b_dinov2']; SOURCES = ['HAM10000', 'BCN20000']
VARIANTS = [(s, a) for s in SOURCES for a in ARCHS]
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
def ci(v): v = np.asarray(v, float); v = v[np.isfinite(v)]; return [round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4)]
RNG = np.random.default_rng(20261004); NB = 1000
SETS = {}
for es in ('HIBA', 'PAD'):
    D = pd.read_csv(f'{DERM}/manifest_{es}.csv', dtype={'image_id': str, 'patient_id': str}).sort_values('image_id')
    D = D[~D.duplicated('raw_sha256', keep='first') & D.dx.isin(CLASSES8)].reset_index(drop=True)
    P = {(s, a, k): pd.read_csv(f'{DERM}/predictions/{es}__{s}_{a}_seed{k}.csv', dtype={'image_id': str}).set_index('image_id').p_malignant.reindex(D.image_id).to_numpy()
         for s in SOURCES for a in ARCHS for k in (0, 1)}
    mal = D.dx.isin(['MEL', 'BCC', 'SCC']).to_numpy(); cl = pd.factorize(D.patient_id.astype(str))[0]; ncl = cl.max() + 1
    SETS[es] = {'P': P, 'mal': mal, 'mel': D.dx.eq('MEL').to_numpy(), 'f': 1.31 * mal.mean(),
                'W': [np.bincount(RNG.integers(0, ncl, ncl), minlength=ncl)[cl].astype(float) for _ in range(NB)]}
    print(es, len(D), 'images')

Mounted at /content/drive
HIBA 1616 images
PAD 2283 images


In [2]:
# Cell 2: split by source, adjusted effect, AUCs
def grp(va, vb):
    if va[0] != vb[0]: return 'mixed'
    return 'same_' + va[0]
OUT = {}
for es, tname in (('HIBA', 'mel'), ('HIBA', 'mal'), ('PAD', 'mal')):
    S = SETS[es]; tgt = S[tname]; R = {k: refer(v, S['f']) for k, v in S['P'].items()}; VEC = {}
    for i, va in enumerate(VARIANTS):
        for vb in VARIANTS[i + 1:]:
            if va[1] == vb[1]: continue
            ma = mb = both = 0
            for ka in (0, 1):
                for kb in (0, 1):
                    x, y = tgt & ~R[(*va, ka)], tgt & ~R[(*vb, kb)]; ma = ma + x; mb = mb + y; both = both + (x & y)
            VEC[(va, vb)] = (ma / 4, mb / 4, both / 4)
    def stats(w):
        T = tgt @ w; g = {'same_HAM10000': [], 'same_BCN20000': [], 'mixed': []}; rows = []
        for p, (ma, mb, both) in VEC.items():
            a_, b_, c_ = ma @ w, mb @ w, both @ w
            if a_ * b_ <= 0: continue
            r = c_ / (a_ * b_ / T); g[grp(*p)].append(r); rows.append((r, float(p[0][0] == p[1][0]), (a_ + b_) / 2, abs(a_ - b_)))
        m = {k: float(np.mean(v)) for k, v in g.items()}
        m['HAM_pairs_minus_mixed'] = m['same_HAM10000'] - m['mixed']; m['BCN_pairs_minus_mixed'] = m['same_BCN20000'] - m['mixed']
        X = np.array(rows); Z = np.column_stack([np.ones(len(X)), X[:, 1], X[:, 2] / X[:, 2].mean(), X[:, 3] / max(X[:, 3].mean(), 1e-9)])
        m['same_source_adjusted_for_miss_counts'] = float(np.linalg.lstsq(Z, X[:, 0], rcond=None)[0][1]); return m
    obs = stats(np.ones(len(tgt))); bs = [stats(w) for w in S['W']]
    OUT[f'{es}_{"melanoma" if tname == "mel" else "any_cancer"}'] = {k: {'mean': round(v, 3), 'ci95': ci([b[k] for b in bs])} for k, v in obs.items()}
AUCS = {f'{es}_{src}': round(float(np.median([roc_auc_score(SETS[es]['mal'], SETS[es]['P'][(src, a, k)]) for a in ARCHS for k in (0, 1)])), 4) for es in SETS for src in SOURCES}
res = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'plan': 'derm_source_symmetry_plan.json', 'status': 'exploratory',
       'median_auc_any_cancer_by_training_source': AUCS, 'split_by_source': OUT}
json.dump(res, open(f'{DERM}/derm_source_symmetry_results.json', 'w'), indent=2)
drive.flush_and_unmount(); drive.mount('/content/drive')
print(json.dumps(res, indent=1))

Mounted at /content/drive
{
 "generated_utc": "2026-10-03T04:22:26.567892+00:00",
 "plan": "derm_source_symmetry_plan.json",
 "status": "exploratory",
 "median_auc_any_cancer_by_training_source": {
  "HIBA_HAM10000": 0.8809,
  "HIBA_BCN20000": 0.8725,
  "PAD_HAM10000": 0.7446,
  "PAD_BCN20000": 0.7012
 },
 "split_by_source": {
  "HIBA_melanoma": {
   "same_HAM10000": {
    "mean": 3.581,
    "ci95": [
     3.0181,
     4.2793
    ]
   },
   "same_BCN20000": {
    "mean": 3.169,
    "ci95": [
     2.6788,
     3.8322
    ]
   },
   "mixed": {
    "mean": 2.93,
    "ci95": [
     2.4426,
     3.4903
    ]
   },
   "HAM_pairs_minus_mixed": {
    "mean": 0.65,
    "ci95": [
     0.37,
     1.0698
    ]
   },
   "BCN_pairs_minus_mixed": {
    "mean": 0.239,
    "ci95": [
     -0.0588,
     0.7181
    ]
   },
   "same_source_adjusted_for_miss_counts": {
    "mean": 0.445,
    "ci95": [
     0.2429,
     0.804
    ]
   }
  },
  "HIBA_any_cancer": {
   "same_HAM10000": {
    "mean": 4.383,
   